# 🚀 TFM - Benchmarking en Google Colab GPU

## Sistema completo de captura de métricas

**IMPORTANTE**: Antes de ejecutar:
1. **Runtime → Change Runtime Type → GPU (T4 o superior)**
2. Guardar configuración
3. Ejecutar celdas en orden

---

### 📊 Datos que se capturarán:
- Información del dispositivo (GPU Colab)
- Tiempos de entrenamiento
- Throughput (samples/segundo)
- Métricas del modelo
- CSVs detallados para comparativa

## 📦 Paso 1: Instalar Dependencias

In [ ]:
# Instalar dependencias (solo si faltan)
!pip install -q tensorflow pandas matplotlib scikit-learn numpy

print("✓ Dependencias instaladas")

## 🔧 Paso 2: Cargar Sistema de Benchmarking

In [ ]:
%%writefile benchmark_utils.py
"""benchmark_utils.py - Versión simplificada para Colab"""

import tensorflow as tf
import platform
import os
from datetime import datetime
import numpy as np
import pandas as pd

def get_device_info():
    """Captura información del dispositivo en Colab."""
    info = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'platform': 'Colab',
        'cpu_cores': os.cpu_count() or 0,
        'tensorflow_version': tf.__version__,
        'device_type': 'CPU',
        'device_name': 'CPU',
        'gpu_count': 0,
        'gpu_names': [],
    }
    
    # Detectar GPUs
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        info['gpu_count'] = len(gpus)
        info['device_type'] = 'Colab_GPU'
        info['gpu_names'] = [gpu.name for gpu in gpus]
        info['device_name'] = gpus[0].name if gpus else 'GPU'
    
    return info

def calculate_model_metrics(model):
    """Calcula métricas del modelo."""
    total_params = model.count_params()
    trainable_params = sum([tf.reduce_prod(var.shape).numpy() 
                           for var in model.trainable_variables])
    
    return {
        'total_parameters': int(total_params),
        'trainable_parameters': int(trainable_params),
        'layers_count': len(model.layers)
    }

def enrich_results(base_results, history, model, device_info, 
                   dataset_size_train=0, dataset_size_test=0):
    """Enriquece resultados con métricas detalladas."""
    enriched = base_results.copy()
    
    # Info del dispositivo
    enriched.update({
        'timestamp': device_info['timestamp'],
        'device_type': device_info['device_type'],
        'device_name': device_info['device_name'],
        'gpu_count': device_info['gpu_count'],
        'cpu_cores': device_info['cpu_cores'],
        'tensorflow_version': device_info['tensorflow_version'],
    })
    
    # Métricas del modelo
    model_metrics = calculate_model_metrics(model)
    enriched.update(model_metrics)
    
    # Métricas de entrenamiento
    if history and hasattr(history, 'history'):
        epochs_executed = len(history.history.get('loss', []))
        enriched['epochs_executed'] = epochs_executed
        
        if 'val_accuracy' in history.history:
            best_epoch = np.argmax(history.history['val_accuracy']) + 1
            enriched['best_epoch'] = int(best_epoch)
            enriched['best_val_accuracy'] = float(max(history.history['val_accuracy']))
            enriched['best_val_loss'] = float(min(history.history['val_loss']))
    
    # Métricas temporales
    training_time = base_results.get('training_time', 0)
    epochs_executed = enriched.get('epochs_executed', 1)
    if training_time > 0 and epochs_executed > 0:
        enriched['time_per_epoch'] = round(training_time / epochs_executed, 2)
    
    # Dataset sizes
    if dataset_size_train > 0:
        enriched['dataset_size_train'] = dataset_size_train
        enriched['samples_per_second'] = round(dataset_size_train / training_time, 2)
    
    if dataset_size_test > 0:
        enriched['dataset_size_test'] = dataset_size_test
    
    return enriched

def print_device_summary(device_info):
    """Imprime resumen del dispositivo."""
    print("\n" + "="*70)
    print("INFORMACIÓN DEL SISTEMA")
    print("="*70)
    print(f"Timestamp: {device_info['timestamp']}")
    print(f"Platform: {device_info['platform']}")
    print(f"CPU Cores: {device_info['cpu_cores']}")
    print(f"TensorFlow: {device_info['tensorflow_version']}")
    print(f"\nDispositivo de Cómputo: {device_info['device_type']}")
    
    if device_info['gpu_count'] > 0:
        print(f"GPUs Detectadas: {device_info['gpu_count']}")
        for i, gpu_name in enumerate(device_info['gpu_names'], 1):
            print(f"  GPU {i}: {gpu_name}")
    else:
        print("Modo: CPU")
    
    print("="*70)

print("✓ benchmark_utils.py creado")

## 🔍 Paso 3: Verificar GPU y Capturar Info

In [ ]:
import tensorflow as tf
from benchmark_utils import get_device_info, print_device_summary

# Capturar información del dispositivo
device_info = get_device_info()
print_device_summary(device_info)

# Verificación adicional
print("\n" + "="*70)
print("VERIFICACIÓN DETALLADA")
print("="*70)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ {len(gpus)} GPU(s) detectada(s)")
    for gpu in gpus:
        print(f"   {gpu}")
    print(f"\n✅ CUDA compilado: {tf.test.is_built_with_cuda()}")
    print(f"✅ GPU disponible: {tf.test.is_gpu_available()}")
else:
    print("⚠️  No se detectó GPU")
    print("   → Runtime → Change Runtime Type → GPU")

print("="*70)

## 🏗️ Paso 4: Preparar Experimento (CNN o LSTM)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from benchmark_utils import enrich_results, calculate_model_metrics
import time

print("✓ Librerías importadas")

## 📊 Paso 5: Experimento - Fashion MNIST (CNN)

In [ ]:
print("=" * 70)
print("EXPERIMENTO: Fashion MNIST con CNN")
print("=" * 70)

# Cargar dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Convertir a RGB
x_train = np.repeat(x_train, 3, axis=-1)
x_test = np.repeat(x_test, 3, axis=-1)

# Split train/val
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

# Crear modelo CNN
model = models.Sequential([
    layers.Input(shape=(28, 28, 3)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModelo creado:")
model.summary()

# Entrenar
print("\n" + "=" * 70)
print("ENTRENANDO...")
print("=" * 70)

start_time = time.time()

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=10,  # Ajustar según necesites
    batch_size=32,
    verbose=1
)

training_time = time.time() - start_time

# Evaluar
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

print("\n" + "=" * 70)
print("RESULTADOS")
print("=" * 70)
print(f"Accuracy: {accuracy:.4f}")
print(f"Loss: {loss:.4f}")
print(f"Training Time: {training_time:.2f} segundos")
print("=" * 70)

# Enriquecer resultados
base_results = {
    'dataset': 'Fashion MNIST',
    'accuracy': accuracy,
    'loss': loss,
    'training_time': training_time
}

enriched_results = enrich_results(
    base_results=base_results,
    history=history,
    model=model,
    device_info=device_info,
    dataset_size_train=len(x_train),
    dataset_size_test=len(x_test)
)

# Guardar CSV
df_results = pd.DataFrame([enriched_results])
df_results.to_csv('fashion_mnist_colab_detallado.csv', index=False)

print("\n✓ Resultados guardados en: fashion_mnist_colab_detallado.csv")
print("\nMétricas capturadas:")
print(df_results.T)

## 📉 Paso 6: Visualizar Resultados

In [ ]:
# Gráfico de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_title('Fashion MNIST - Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Fashion MNIST - Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fashion_mnist_training.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: fashion_mnist_training.png")

## 💾 Paso 7: Descargar Resultados

In [ ]:
from google.colab import files

print("Descargando archivos...")

# Descargar CSV detallado
files.download('fashion_mnist_colab_detallado.csv')
print("✓ CSV descargado")

# Descargar gráfico
files.download('fashion_mnist_training.png')
print("✓ Gráfico descargado")

print("\n" + "="*70)
print("✓ EXPERIMENTO COMPLETADO")
print("="*70)
print("\nArchivos descargados:")
print("  1. fashion_mnist_colab_detallado.csv")
print("  2. fashion_mnist_training.png")
print("\nPróximos pasos:")
print("  - Copia el CSV a tu carpeta local TFM_Fase1/csv_data/")
print("  - Ejecuta: python CODE/utils/comparador_dispositivos.py")
print("  - Genera comparativas CPU vs Colab GPU")

---

## 📋 Resumen

Este notebook captura:
- ✅ Información del dispositivo Colab (GPU tipo T4, V100, A100)
- ✅ Tiempos de entrenamiento detallados
- ✅ Throughput (samples/segundo)
- ✅ Métricas del modelo (parámetros, layers)
- ✅ Mejor epoch y métricas de validación
- ✅ CSV detallado para comparativa

### Comparar con local:
1. Descarga el CSV de Colab
2. Ejecútalo localmente (CPU o GPU)
3. Usa `comparador_dispositivos.py` para gráficos

### Speedup esperado Colab GPU vs CPU local:
- Fashion MNIST: ~6-8x más rápido
- CIFAR-10: ~8-10x más rápido
- ECG5000 LSTM: ~3-5x más rápido
- UCI HAR LSTM: ~3-5x más rápido